# 02 — LLM Insight Generation

Validate the LLM insight generation pipeline end-to-end:
- Load analytics from S3 (written by `01_umami_ingest`)
- Call `generate_insights()` — the same production function the agent runs daily
- Works with any provider: set `LLM_PROVIDER=ionos` or `LLM_PROVIDER=mistral` in `.env`

In [ ]:
import os
prefix = os.getenv("S3_STATE_PREFIX", "growth-agent/")
if prefix == "growth-agent/":
    raise RuntimeError(
        f"S3_STATE_PREFIX is {prefix!r} — this is the PRODUCTION prefix. "
        "Set S3_STATE_PREFIX=growth-agent-dev/ in .env before running notebooks."
    )
print(f"Using S3 prefix: {prefix}  ✓")


In [13]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os
import logging

# Surface log output from production code
logging.basicConfig(level=logging.INFO, format='%(name)s %(levelname)s: %(message)s')

load_dotenv('../.env', override=True)

# Set provider here — picked up by generate_insights() via os.environ
os.environ['LLM_PROVIDER'] = os.environ.get('LLM_PROVIDER', 'ionos')  # change to 'mistral' if needed

from agent.llm_client import PROVIDERS
provider = os.environ['LLM_PROVIDER']
config = PROVIDERS.get(provider)
key_env = config.api_key_env if config else 'LLM_PROVIDER'
api_key_set = bool(os.environ.get(key_env))

print(f'Active provider : {provider}')
print(f'API key loaded  : {"yes" if api_key_set else f"NO — set {key_env} in .env"}')

Active provider : ionos
API key loaded  : yes


## 1. Load Analytics Data

Load the insights from `01_umami_ingest` notebook output.

In [14]:
from agent.storage import S3Storage

store = S3Storage(
    bucket=os.getenv('S3_BUCKET'),
    prefix=os.getenv('S3_STATE_PREFIX', 'growth-agent/'),
    access_key=os.getenv('SCW_ACCESS_KEY'),
    secret_key=os.getenv('SCW_SECRET_KEY'),
)
insights_raw = store.read('insights.json')

if insights_raw is None:
    print('ERROR: Run 01_umami_ingest.ipynb first to generate insights.json')
else:
    print(f'Loaded insights : {insights_raw["website_analytics"]["visitors"]} visitors')
    print(f'Top pages       : {len(insights_raw["website_analytics"]["top_pages"])}')

Loaded insights : 43 visitors
Top pages       : 10


## 2. Run Production Insight Generation

Calls the same `generate_insights()` function the agent uses in production.
The LLM provider and model are picked up from `LLM_PROVIDER` / `LLM_MODEL` env vars.

In [15]:
from agent.nodes.insights import generate_insights

# Raises on failure — check provider and API key above if this fails
analysis = generate_insights(store)

print(f'Provider        : {provider}')
print(f'Top topics      : {analysis.top_topics}')
print(f'Pages for social: {len(analysis.best_pages_for_social)}')

httpx INFO: HTTP Request: GET https://fretchen.eu/ "HTTP/1.1 301 Moved Permanently"
httpx INFO: HTTP Request: GET https://www.fretchen.eu "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://fretchen.eu/amo/13/ "HTTP/1.1 301 Moved Permanently"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/amo/13/ "HTTP/1.1 404 Not Found"
httpx INFO: HTTP Request: GET https://fretchen.eu/amo/10 "HTTP/1.1 301 Moved Permanently"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/amo/10 "HTTP/1.1 404 Not Found"
httpx INFO: HTTP Request: GET https://fretchen.eu/blog/ "HTTP/1.1 301 Moved Permanently"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://fretchen.eu/de/blog/ "HTTP/1.1 301 Moved Permanently"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/de/blog/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://fretchen.eu/amo/11 "HTTP/1.1 301 Moved Permanently"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/amo/11

Provider        : ionos
Top topics      : ['Politische Ökonomie & Spieltheorie', 'Blockchain & Web3']
Pages for social: 10


## 4. Inspect Structured Output

In [16]:
# analysis is already a typed LLMAnalysis — no JSON parsing needed
print('=== Parsed Analysis ===')
print(f'\nTop topics: {analysis.top_topics}')
print(f'\nTraffic sources: {analysis.traffic_sources}')
print(f'\nBest pages for social:')
for page in analysis.best_pages_for_social:
    print(f'  - {page.url} — {page.reason}')
print(f'\nContent gaps: {analysis.content_gaps}')
print(f'\nGrowth opportunities:')
for opp in analysis.growth_opportunities:
    print(f'  - {opp}')

=== Parsed Analysis ===

Top topics: ['Politische Ökonomie & Spieltheorie', 'Blockchain & Web3']

Traffic sources: ['Direct traffic', 'Bluesky']

Best pages for social:
  - /blog/27/ — Strong audience interest
  - /blog/12/ — Relevant topic with existing engagement
  - /quantum/hardware/ — Underrepresented topic worth exploring
  - /amo/13/ — Mystery page with some traffic
  - /de/blog/27/ — Strong audience interest in other languages
  - /blog/ — General blog page with some traffic
  - /de/blog/ — General blog page with some traffic in other languages
  - /amo/10 — Some traffic, but unclear content
  - /amo/11 — Some traffic, but unclear content
  - /blog/12/ — Relevant topic with existing engagement

Content gaps: ['Quantencomputing & QML', 'AI-Tools & Infrastruktur']

Growth opportunities:
  - Share the lecture series on quantum hardware platforms on Mastodon and Bluesky to attract more visitors interested in Quantencomputing & QML
  - Create a social media post about the AI image g

## 5. Verify S3 Persistence

In [17]:
# generate_insights() already saved insights.json and llm_analysis.json to S3 — just verify
saved = store.read('llm_analysis.json')
print(f'llm_analysis.json saved: {saved is not None}')
print(f'Growth opportunities: {len(saved.get("growth_opportunities", []))}')

llm_analysis.json saved: True
Growth opportunities: 5


In [18]:
print('Done — LLM insight generation validated.')

Done — LLM insight generation validated.
